# 12_validate_novelty_vs_kelly

**Objective:** Validate the present novelty measure against the text-based novelty score of Kelly et al. (2021). Kelly patents are joined to the sample's focal novelty via the BigQuery publication-to-family export, and the distance-based novelty is correlated with Kelly's backward-similarity columns (opposite signs imply a small negative correlation).

**Inputs:** `kelly_matched.parquet`; `novelty_features.parquet`; `lens_to_familyid.parquet`; `bq-results*.csv` (BigQuery publication export).

**Outputs:** `novelty_vs_kelly.parquet`.

In [ ]:
# Imports
import glob
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

In [ ]:
# Paths and settings
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").exists())
RAW  = ROOT / "data" / "raw"
PROC = ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)
KELLY    = RAW / "benchmarks" / "kelly_matched.parquet"
NOVELTY  = PROC / "novelty_features.parquet"
MAPPING  = PROC / "lens_to_familyid.parquet"
OUT_VS   = PROC / "novelty_vs_kelly.parquet"

def _find_bq_pub():
    for f in sorted(glob.glob(str(RAW / "bigquery" / "bq-results*.csv"))):
        head = pd.read_csv(f, nrows=0)
        if "publication_number" in head.columns and "family_id" in head.columns:
            return f
    raise FileNotFoundError("No bq-results*.csv with publication_number/family_id found")

KELLY_SIM_COLS = ["bsim5", "fsim01", "fsim25", "fsim610", "lqsim05", "lqsim010"]
OUR_MEASURES   = ["novelty_mean", "novelty_top10", "novelty_min"]

## Map Kelly patents to DOCDB families

In [ ]:
# Build a US patent_num -> family_id lookup from the BigQuery publication export
def us_num_to_family():
    bq = pd.read_csv(_find_bq_pub(), dtype=str,
                     usecols=["family_id", "country_code", "publication_number"])
    us = bq[bq["country_code"] == "US"].copy()
    us["patent_num"] = us["publication_number"].str.extract(r"US-?0*(\d+)-")[0]
    us = us.dropna(subset=["patent_num", "family_id"]).drop_duplicates("patent_num")
    us["family_id"] = pd.to_numeric(us["family_id"], errors="coerce")
    return us[["patent_num", "family_id"]]

## Merge and correlate

In [ ]:
# Join Kelly measures to the present novelty and report Spearman correlations
def correlate():
    km = pd.read_parquet(KELLY)
    km["patent_num"] = km["patent_num"].astype(str).str.strip()

    nov = pd.read_parquet(NOVELTY, columns=["lens_id"] + OUR_MEASURES)
    l2f = (pd.read_parquet(MAPPING, engine="fastparquet")[["lens_id", "docdb_family_id"]]
             .dropna(subset=["docdb_family_id"]).drop_duplicates("lens_id"))
    nov_fam = (nov.merge(l2f, on="lens_id", how="inner")
                  .drop_duplicates("docdb_family_id"))

    us = us_num_to_family()
    df = (km.merge(us, on="patent_num", how="inner")
            .merge(nov_fam, left_on="family_id", right_on="docdb_family_id", how="inner"))
    df.to_parquet(OUT_VS)
    print(f"  matched Kelly patents with novelty: n={df['patent_num'].nunique():,}  -> {OUT_VS.name}")
    print("  (n may differ slightly from the thesis depending on family de-duplication)")

    d = df[["novelty_mean", "bsim5"]].dropna()
    rho, p = spearmanr(d["novelty_mean"], d["bsim5"])
    print(f"\n=== HEADLINE  novelty_mean vs Kelly bsim5 ===")
    print(f"  n={len(d):,}  Spearman rho={rho:+.3f}  (p={p:.2e})   [thesis: rho=-0.10]")

    print("\n=== Spearman rho: present measures vs Kelly columns ===")
    header = "  {:14s}".format("") + "".join(f"{c:>10s}" for c in KELLY_SIM_COLS)
    print(header)
    for our in OUR_MEASURES:
        cells = []
        for c in KELLY_SIM_COLS:
            dd = df[[our, c]].dropna()
            cells.append(f"{spearmanr(dd[our], dd[c])[0]:+.3f}" if len(dd) > 50 else "    n/a")
        print(f"  {our:14s}" + "".join(f"{v:>10s}" for v in cells))
    return df

In [ ]:
# Entry point
if __name__ == "__main__":
    correlate()